In [1]:
import os
# 路径指向你的 sports 目录
data_dir = '/root/autodl-tmp/PromptMM/data/clothing/'
# 建议先检查 5-core 子目录
json_dir = os.path.join(data_dir, '5-core')

In [2]:
def check_files():
    print("--- 1. 文件存在性检查 ---")
    required_json = ['train.json', 'test.json']
    required_npy = ['text_feat.npy', 'image_feat.npy']
    
    for f in required_json:
        path = os.path.join(json_dir, f)
        print(f"[{'OK' if os.path.exists(path) else 'MISSING'}] {path}")
    for f in required_npy:
        path = os.path.join(data_dir, f)
        print(f"[{'OK' if os.path.exists(path) else 'MISSING'}] {path}")

    print("\n--- 2. 多模态特征维度检查 ---")
    t_feat = np.load(os.path.join(data_dir, 'text_feat.npy'))
    v_feat = np.load(os.path.join(data_dir, 'image_feat.npy'))
    print(f"文本特征形状 (Items, Dim): {t_feat.shape}")
    print(f"图像特征形状 (Items, Dim): {v_feat.shape}")

    print("\n--- 3. 交互数据 (JSON) 统计 ---")
    all_users = set()
    all_items = set()
    
    for split in ['train.json', 'test.json']:
        path = os.path.join(json_dir, split)
        if not os.path.exists(path): continue
        with open(path, 'r') as f:
            data = json.load(f)
            user_count = len(data)
            inter_count = 0
            for u, items in data.items():
                all_users.add(int(u))
                for i in items:
                    all_items.add(int(i))
                    inter_count += 1
            print(f"{split}: 用户数={user_count}, 交互数={inter_count}")

    print("\n--- 4. 关键参数确认 ---")
    n_user = max(all_users) + 1
    n_item = max(all_items) + 1
    print(f"最大用户 ID + 1 (n_user): {n_user}")
    print(f"最大物品 ID + 1 (n_item): {n_item}")
    
    # 核心检查：特征数量是否覆盖了交互中的所有物品
    if n_item > v_feat.shape[0]:
        print(f"❌ 警告: 物品最大 ID ({n_item-1}) 超过了特征行数 ({v_feat.shape[0]})！")
        print("这会导致训练时 Embedding 报错，需要检查 ID 是否连续或是否进行了重编号。")
    else:
        print("✅ 特征行数能够覆盖所有物品 ID。")

In [3]:
import numpy as np
import json
import os

In [4]:
if __name__ == "__main__":
    check_files()

--- 1. 文件存在性检查 ---
[OK] /root/autodl-tmp/PromptMM/data/clothing/5-core/train.json
[OK] /root/autodl-tmp/PromptMM/data/clothing/5-core/test.json
[OK] /root/autodl-tmp/PromptMM/data/clothing/text_feat.npy
[OK] /root/autodl-tmp/PromptMM/data/clothing/image_feat.npy

--- 2. 多模态特征维度检查 ---
文本特征形状 (Items, Dim): (23033, 1024)
图像特征形状 (Items, Dim): (23033, 4096)

--- 3. 交互数据 (JSON) 统计 ---
train.json: 用户数=39387, 交互数=197338
test.json: 用户数=39387, 交互数=40150

--- 4. 关键参数确认 ---
最大用户 ID + 1 (n_user): 39387
最大物品 ID + 1 (n_item): 23033
✅ 特征行数能够覆盖所有物品 ID。
